# 적용하기 — 보도자료 → 기사 첫 문장(리드) 학습 데이터
1. 수집 `press_raw.csv` → 2. 정제 `press_clean.csv` → 3. 리드 초안 `pairs_draft.csv` → 4. `lead_draft.jsonl`

⚠️ 수집 전: robots.txt · 이용약관 · 공공누리 표시 확인. CSV·JSONL은 GitHub에 올리지 않아요.

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → CPU로 실행 (이 노트북은 CPU로도 충분해요)')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

In [ ]:
# ▶ API 키 불러오기 — Colab은 🔑 보안 비밀, VS Code는 같은 폴더의 .env 파일
#   .env 파일 내용 예:  OPENAI_API_KEY=sk-...   (GitHub에 올리지 않기)
def get_key(name='OPENAI_API_KEY'):
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        print('pip install python-dotenv 후 다시 실행')
    return os.getenv(name)

OPENAI_API_KEY = get_key()
print('API 키:', '준비됨' if OPENAI_API_KEY else '없음 → 키를 등록하세요')

## 1. 수집 — 주황 칸 4개를 F12로 찾아 바꾸기

In [ ]:
import requests, time, pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

LIST_URL = '목록 주소 (페이지 번호 자리는 {})'
LINK, TITLE, BODY = '목록 링크 선택자', '제목 선택자', '본문 선택자'
H = {'User-Agent': 'Mozilla/5.0'}
rows = []
for page in range(1, 11):      # 먼저 range(1, 2)로 한 페이지만 시험
    soup = BeautifulSoup(requests.get(LIST_URL.format(page), headers=H).text, 'html.parser')
    for a in soup.select(LINK):
        d = BeautifulSoup(requests.get(urljoin(LIST_URL, a['href']), headers=H).text, 'html.parser')
        rows.append({'제목': d.select_one(TITLE).get_text(strip=True),
                     '본문': d.select_one(BODY).get_text(' ', strip=True)})
        time.sleep(1)
pd.DataFrame(rows).to_csv('press_raw.csv', index=False, encoding='utf-8-sig')
print(len(rows), '건 수집')

## 2. 정제 — 공백 정리 · 빈 본문 · 중복 · 길이(200~2,500자)

In [ ]:
df = pd.read_csv('press_raw.csv')
print('처음:', len(df))

df['본문'] = df['본문'].str.replace(r'\s+', ' ', regex=True).str.strip()
df = df.dropna(subset=['본문'])
df = df.drop_duplicates(subset=['본문'])
df = df[df['본문'].str.len().between(200, 2500)]

print('정제 후:', len(df))
df.to_csv('press_clean.csv', index=False, encoding='utf-8-sig')

## 3. 리드 초안 — GPT가 쓰고, 확정은 사람이
키 준비: Colab은 왼쪽 🔑(보안 비밀)에 `OPENAI_API_KEY` 등록 → 노트북 액세스 켜기 / VS Code는 `.env` 파일

In [ ]:
from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def draft_lead(body):
    prompt = f"""다음 보도자료로 기사 첫 문장(리드)을 한 문장으로 써.
- 보도자료에 없는 숫자·원인·전망은 쓰지 마.
- 누가·언제·무엇을 담아.
보도자료: {body[:2000]}"""
    r = client.chat.completions.create(model='gpt-4.1-mini',
                                       messages=[{'role': 'user', 'content': prompt}])
    return r.choices[0].message.content.strip()

df = pd.read_csv('press_clean.csv')
print(draft_lead(df['본문'].iloc[0]))      # 먼저 1개로 시험

In [ ]:
df['리드'] = df['본문'].apply(draft_lead)
df.to_csv('pairs_draft.csv', index=False, encoding='utf-8-sig')

## 4. JSONL로 저장 — 한 줄 = 한 쌍 (alpaca 형식)

In [ ]:
import json
INST = '보도자료를 읽고 기사 첫 문장(리드)을 한 문장으로 쓰세요.'
with open('lead_draft.jsonl', 'w', encoding='utf-8') as f:
    for _, r in df.iterrows():
        row = {'instruction': INST, 'input': r['본문'], 'output': r['리드']}
        f.write(json.dumps(row, ensure_ascii=False) + '\n')
print(open('lead_draft.jsonl', encoding='utf-8').readline()[:200])

다음 단계: `pairs_draft.csv`를 엑셀로 열어 **판정(사용·수정·제외)** 칸을 채우고 `pairs_checked.csv`로 저장